In [ ]:
import numpy as np
import librosa
from scipy.io import wavfile
from sklearn.preprocessing import MinMaxScaler
import scipy.fftpack
import matplotlib.pyplot as plt
import pandas as pd
import os
from tqdm import tqdm

In [ ]:
# 전처리 및 특징 추출 파라미터
sampling_rate = 16000  # 샘플링 속도
fft_size = 1024  # FFT 사이즈
window_length = int(0.025 * sampling_rate)  # 윈도우 길이 (25ms)
hop_length = int(0.010 * sampling_rate)  # 홉 길이 (10ms)
n_mfcc = 100

In [ ]:
def load_file(filename, sampling_rate=16000):
    data, sr = librosa.load(filename, sr=sampling_rate)
    return data, sr

In [ ]:
def remove_silence(y, sr, top_db=20):
    # 무음 구간의 시작과 끝을 찾기
    intervals = librosa.effects.split(y, top_db=top_db)

    # 유효한 오디오 신호를 합치기
    y_trimmed = np.concatenate([y[start:end] for start, end in intervals])

    return y_trimmed

In [ ]:
def mel_spectrogram_generator(data, sr, fft_size, hop_length):
    mel_spectrogram = librosa.feature.melspectrogram(y=data, sr=sr, n_fft=fft_size, hop_length=hop_length, window='hamming')
    return mel_spectrogram

In [ ]:
def normalize_spectrogram(mel_spectrogram):
    scaler = MinMaxScaler()
    mel_spectrogram_norm = scaler.fit_transform(mel_spectrogram.T).T
    return mel_spectrogram_norm

In [ ]:
def compute_log_mel_spectrogram(mel_spectrogram_norm):
    return np.log(mel_spectrogram_norm + 1e-6)

In [ ]:
def apply_dct(log_mel_spectrogram, n_mfcc):
    return scipy.fftpack.dct(log_mel_spectrogram, type=2, axis=1, norm='ortho')[:, :n_mfcc]

In [ ]:
def plot_mfcc(mfcc, sampling_rate, hop_length, vmin=1, vmax=-1):
    plt.figure(figsize=(10, 6))
    plt.imshow(mfcc.T, aspect='auto', origin='lower', 
               extent=[0, mfcc.shape[0] * hop_length / sampling_rate, 0, mfcc.shape[1]], vmin=vmin, vmax=vmax)
    plt.colorbar()
    plt.title('MFCC')
    plt.xlabel('Time (s)')
    plt.ylabel('MFCC Coefficients')
    plt.tight_layout()
    plt.show()

In [ ]:
def compute_mfcc(filename, sampling_rate=16000, fft_size=1024, window_length=400, hop_length=160, n_mfcc=13):
    data, sr = load_file(filename, sampling_rate)

    # 무음 제거
    data = remove_silence(data, sr)
    
    mel_spectrogram = mel_spectrogram_generator(data, sr, fft_size, hop_length)
    mel_spectrogram_norm = normalize_spectrogram(mel_spectrogram)
    log_mel_spectrogram = compute_log_mel_spectrogram(mel_spectrogram_norm)
    mfcc = apply_dct(log_mel_spectrogram, n_mfcc)
    
    return mfcc

In [ ]:
df = pd.read_csv('../data/train.csv')
file_paths = df['path'].values
labels = df['label'].apply(lambda x: 1 if x == 'real' else 0).values

In [ ]:
mfcc_features = []
for path in tqdm(file_paths):
    mfcc = compute_mfcc(path, sampling_rate, fft_size, window_length, hop_length, n_mfcc)
    mfcc_features.append(mfcc)

In [ ]:
max_length = max(mfcc.shape[0] for mfcc in mfcc_features)
max_length

In [ ]:
max_length = max(mfcc.shape[1] for mfcc in mfcc_features)
max_length

In [ ]:
max_features = n_mfcc 

In [ ]:
padded_mfcc_features = []

for mfcc in tqdm(mfcc_features):
    pad_width = max_length - mfcc.shape[0]
    pad_feature_width = max_features - mfcc.shape[1] if mfcc.shape[1] < max_features else 0
    mfcc = np.pad(mfcc, ((0, pad_width), (0, pad_feature_width)), mode='constant')
    padded_mfcc_features.append(mfcc)

In [ ]:
df['label'] = df['label'].apply(lambda x: 1 if x == 'real' else 0)
labels = df['label'].values

In [ ]:
padded_mfcc_features = np.array(padded_mfcc_features)
labels = np.array(labels)

In [ ]:
from keras.utils import to_categorical

In [ ]:
labels = to_categorical(labels, num_classes=2)

In [ ]:
labels

In [ ]:
np.save('mfcc_features.npy', padded_mfcc_features)
np.save('labels.npy', labels)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical

In [ ]:
mfcc_features = np.load('mfcc_features.npy')
labels = np.load('labels.npy')

In [ ]:
padded_mfcc_features = padded_mfcc_features[..., np.newaxis]

In [ ]:
# labels = to_categorical(labels, )

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_valid, y_train, y_valid = train_test_split(padded_mfcc_features, labels, test_size=0.2, random_state=42)

In [ ]:
from keras.layers import Conv2D , MaxPool2D , Input , GlobalAveragePooling2D ,AveragePooling2D, Dense , Dropout ,Activation , BatchNormalization

In [ ]:
from tensorflow import keras
from keras.models import Model
from tensorflow.keras import layers
from keras.layers import concatenate
from tensorflow.keras.utils import plot_model

In [ ]:
def InceptionV4():
    
    input_layer = Input(shape=(128, 100, 1))
    
    x = stemBlock(prev_layer=input_layer)
    
    x = InceptionBlock_A(prev_layer=x)
    x = InceptionBlock_A(prev_layer=x)
    x = InceptionBlock_A(prev_layer=x)
    x = InceptionBlock_A(prev_layer=x)
    
    x = reduction_A_Block(prev_layer=x)
    
    x = InceptionBlock_B(prev_layer=x)
    x = InceptionBlock_B(prev_layer=x)
    x = InceptionBlock_B(prev_layer=x)
    x = InceptionBlock_B(prev_layer=x)
    x = InceptionBlock_B(prev_layer=x)
    x = InceptionBlock_B(prev_layer=x)
    x = InceptionBlock_B(prev_layer=x)
    
    x = reduction_B_Block(prev_layer= x)
    
    x = InceptionBlock_C(prev_layer=x)
    x = InceptionBlock_C(prev_layer=x)
    x = InceptionBlock_C(prev_layer=x)
    
    x = GlobalAveragePooling2D()(x)
    
    x = Dense(units = 1536, activation='relu') (x)
    x = Dropout(rate = 0.8) (x)
    x = Dense(units = 2, activation='sigmoid')(x)
    
    model = Model(inputs = input_layer , outputs = x , name ='Inception-V4')
    
    return model


def conv2d_with_Batch(prev_layer , nbr_kernels , filter_size , strides = (1,1) , padding = 'same'):
    x = Conv2D(filters = nbr_kernels, kernel_size = filter_size, strides=strides , padding=padding) (prev_layer)
    x = BatchNormalization()(x)
    x = Activation(activation = 'relu') (x)
    return x


def stemBlock(prev_layer):
    x = conv2d_with_Batch(prev_layer, nbr_kernels = 32, filter_size = (3,3), strides = (2,2))
    x = conv2d_with_Batch(x, nbr_kernels = 32, filter_size = (3,3))
    x = conv2d_with_Batch(x, nbr_kernels = 64, filter_size = (3,3))
    
    x_1 = conv2d_with_Batch(x, nbr_kernels = 96, filter_size = (3,3), strides = (2,2) )
    x_2 = MaxPooling2D(pool_size=(3,3) , strides=(2,2), padding='same') (x)
    
    x = concatenate([x_1 , x_2], axis = 3)
    
    x_1 = conv2d_with_Batch(x, nbr_kernels = 64, filter_size = (1,1))
    x_1 = conv2d_with_Batch(x_1, nbr_kernels = 64, filter_size = (1,7) , padding ='same')
    x_1 = conv2d_with_Batch(x_1, nbr_kernels = 64, filter_size = (7,1), padding ='same')
    x_1 = conv2d_with_Batch(x_1, nbr_kernels = 96, filter_size = (3,3))
    
    x_2 = conv2d_with_Batch(x, nbr_kernels = 96, filter_size = (1,1))
    x_2 = conv2d_with_Batch(x_2, nbr_kernels = 96, filter_size = (3,3))
    
    x = concatenate([x_1 , x_2], axis = 3)
    
    x_1 = conv2d_with_Batch(x, nbr_kernels = 192, filter_size = (3,3) , strides=2)
    x_2 = MaxPooling2D(pool_size=(3,3) , strides=(2,2), padding='same') (x)
    
    x = concatenate([x_1 , x_2], axis = 3)
    
    return x


def reduction_A_Block(prev_layer):
    x_1 = conv2d_with_Batch(prev_layer, nbr_kernels=192, filter_size=(1,1))
    x_1 = conv2d_with_Batch(x_1, nbr_kernels=224, filter_size=(3,3), padding='same')
    x_1 = conv2d_with_Batch(x_1, nbr_kernels=256, filter_size=(3,3), strides=(2,2))

    x_2 = conv2d_with_Batch(prev_layer, nbr_kernels=384, filter_size=(3,3), strides=(2,2))

    x_3 = MaxPooling2D(pool_size=(3,3), strides=(2,2), padding='same')(prev_layer)

    x = concatenate([x_1, x_2, x_3], axis=3)

    return x



def reduction_B_Block(prev_layer):
    x_1 = MaxPooling2D(pool_size=(3,3), strides=(1,1), padding='same')(prev_layer)
    
    x_2 = conv2d_with_Batch(prev_layer=prev_layer, nbr_kernels=192, filter_size=(1,1))
    x_2 = conv2d_with_Batch(prev_layer=x_2, nbr_kernels=192, filter_size=(3,3), strides=(1,1), padding='same')
    
    x_3 = conv2d_with_Batch(prev_layer=prev_layer, nbr_kernels=256, filter_size=(1,1))
    x_3 = conv2d_with_Batch(prev_layer=x_3, nbr_kernels=256, filter_size=(1,7), padding='same')
    x_3 = conv2d_with_Batch(prev_layer=x_3, nbr_kernels=320, filter_size=(7,1), padding='same')
    x_3 = conv2d_with_Batch(prev_layer=x_3, nbr_kernels=320, filter_size=(3,3), strides=(1,1), padding='same')
    
    x = concatenate([x_1, x_2, x_3], axis=3)
    return x



def InceptionBlock_A(prev_layer):
    
    x_1 = conv2d_with_Batch(prev_layer = prev_layer, nbr_kernels = 64, filter_size = (1,1))
    x_1 = conv2d_with_Batch(prev_layer = x_1, nbr_kernels = 96, filter_size = (3,3) , strides=(1,1), padding='same' )
    x_1 = conv2d_with_Batch(prev_layer = x_1, nbr_kernels = 96, filter_size = (3,3) , strides=(1,1) , padding='same')
    
    x_2 = conv2d_with_Batch(prev_layer = prev_layer, nbr_kernels = 64, filter_size = (1,1))
    x_2 = conv2d_with_Batch(prev_layer = x_2, nbr_kernels = 96, filter_size = (3,3) , padding='same')
    
    x_3 = AveragePooling2D(pool_size=(3,3) , strides=1 , padding='same')(prev_layer)
    x_3 = conv2d_with_Batch(prev_layer = x_3, nbr_kernels = 96, filter_size = (1,1) , padding='same')
    
    x_4 = conv2d_with_Batch(prev_layer = prev_layer, nbr_kernels = 96, filter_size = (1,1))
    
    output = concatenate([x_1 , x_2 , x_3 , x_4], axis = 3)

    return output


def InceptionBlock_B(prev_layer):
    
    x_1 = conv2d_with_Batch(prev_layer = prev_layer, nbr_kernels = 192, filter_size = (1,1))
    x_1 = conv2d_with_Batch(prev_layer = x_1, nbr_kernels = 192, filter_size = (7,1) , padding='same')
    x_1 = conv2d_with_Batch(prev_layer = x_1, nbr_kernels = 224, filter_size = (1,7) , padding='same')
    x_1 = conv2d_with_Batch(prev_layer = x_1, nbr_kernels = 224, filter_size = (7,1) , padding='same')
    x_1 = conv2d_with_Batch(prev_layer = x_1, nbr_kernels = 256, filter_size = (1,7), padding='same')
    
    x_2 = conv2d_with_Batch(prev_layer = prev_layer, nbr_kernels = 192, filter_size = (1,1))
    x_2 = conv2d_with_Batch(prev_layer = x_2, nbr_kernels = 192, filter_size = (1,7) , padding='same')
    x_2 = conv2d_with_Batch(prev_layer = x_2, nbr_kernels = 224, filter_size = (7,1), padding='same')
    x_2 = conv2d_with_Batch(prev_layer = x_2, nbr_kernels = 224, filter_size = (1,7), padding='same')
    x_2 = conv2d_with_Batch(prev_layer = x_2, nbr_kernels = 256, filter_size = (7,1), padding='same')
    
    x_3 = AveragePooling2D(pool_size=(3,3) , strides=1 , padding='same')(prev_layer)
    x_3 = conv2d_with_Batch(prev_layer = x_3, nbr_kernels = 128, filter_size = (1,1))
    
    x_4 = conv2d_with_Batch(prev_layer = prev_layer, nbr_kernels = 384, filter_size = (1,1))

    output = concatenate([x_1 , x_2 ,x_3, x_4], axis = 3) 
    return output


def InceptionBlock_C(prev_layer):
    
    x_1 = conv2d_with_Batch(prev_layer = prev_layer, nbr_kernels = 384, filter_size = (1,1))
    x_1 = conv2d_with_Batch(prev_layer = x_1, nbr_kernels = 448, filter_size = (3,1) , padding='same')
    x_1 = conv2d_with_Batch(prev_layer = x_1, nbr_kernels = 512, filter_size = (1,3) , padding='same')
    x_1_1 = conv2d_with_Batch(prev_layer = x_1, nbr_kernels = 256, filter_size = (1,3), padding='same')
    x_1_2 = conv2d_with_Batch(prev_layer = x_1, nbr_kernels = 256, filter_size = (3,1), padding='same')
    x_1 = concatenate([x_1_1 , x_1_2], axis = 3)
    
    x_2 = conv2d_with_Batch(prev_layer = prev_layer, nbr_kernels = 384, filter_size = (1,1))
    x_2_1 = conv2d_with_Batch(prev_layer = x_2, nbr_kernels = 256, filter_size = (1,3), padding='same')
    x_2_2 = conv2d_with_Batch(prev_layer = x_2, nbr_kernels = 256, filter_size = (3,1), padding='same')
    x_2 = concatenate([x_2_1 , x_2_2], axis = 3)
    
    x_3 = MaxPooling2D(pool_size=(3,3),strides = 1 , padding='same')(prev_layer)
    x_3 = conv2d_with_Batch(prev_layer = x_3, nbr_kernels = 256, filter_size = 3  , padding='same')
    
    x_4 = conv2d_with_Batch(prev_layer = prev_layer, nbr_kernels = 256, filter_size = (1,1))
    
    output = concatenate([x_1 , x_2 , x_3 , x_4], axis = 3)
    
    return output

In [ ]:
model = InceptionV4()

In [ ]:
model.build((None, padded_mfcc_features.shape[1], padded_mfcc_features.shape[2], 1))

In [ ]:
model.compile(optimizer='adam',
              loss='categorical_crossentropy',  # 이진 분류의 경우 binary_crossentropy 사용
              metrics=['AUC', 'accuracy'])

In [ ]:
# model.summary()

In [ ]:
# print(f"Shape of x_train: {X_train.shape}")
# print(f"Shape of x_test: {X_valid.shape}")
# print(f"Sample values in x_train: {np.unique(X_train)}")
# print(f"Shape of model input: {model.input_shape}")

In [ ]:
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_valid.shape}")
print(f"Sample values in y_train: {np.unique(y_train)}")
print(f"Shape of model output: {model.output_shape}")

In [ ]:
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2)

In [ ]:
loss, accuracy = model.evaluate(X_valid, y_valid, verbose=0)
print(f'Test Accuracy: {accuracy*100:.2f}%')

In [ ]:
y_pred = model.predict(X_valid)
y_pred_classes = np.argmax(y_pred, axis=1)
# y_true = np.argmax(y_valid, axis=1)

In [ ]:
y_pred

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
conf_matrix = confusion_matrix(y_valid, y_pred_classes)
print('Confusion Matrix:')
print(conf_matrix)

In [ ]:
class_report = classification_report(y_valid, y_pred_classes, target_names=['fake', 'real'])
print('Classification Report:')
print(class_report)

In [ ]:
model.save('cnn_model_test.h5')